# 1. Purpose
In the previous notebook `00_scoping.ipynb`, I've scoped the problem into an initial question: 

>*Did delivery performance (actual vs. estimated delivery date) meaningfully decline in Jan–Mar 2018 compared to the 2017 baseline, and does that decline statistically explain a meaningful share of the drop in average review score over the same period?*

In this notebook, I will be looking at the datasets that are needed for answering this question, in this case, that'll be `orders`, `customers`, `reviews`, and perhaps `order_items`. The exploring, cleaning, and merging will also be conducted here if needed.

In [1]:
# Loading libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sys
from pathlib import Path

sys.path.append("../src")
from helpers import get_connection, sql
pd.set_option('display.max_colwidth', None)

# Loading datasets as tables into SQLite
conn = get_connection(["orders", "reviews", "order_items", "customers"])

## Quick Glimpse on the Datasets

In [ ]:
# First 5 rows of the customers dataset
sql("" \
"SELECT * FROM customers " \
"LIMIT 5;", conn)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
# Total customers and orders
sql("""
SELECT COUNT(DISTINCT customer_unique_id) AS total_customers,
COUNT(DISTINCT customer_id) AS total_orders 
FROM customers
""", conn)

,total_customers,total_orders
0,96096,99441


According to the olist data dictionary (https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data), each order is assigned to an unique `order_id` and `customer_id`, and each order could contain multiple order items.

Since each order is assigned to an unique `customer_id`, I used the count of `customer_id` as the order counts, and the `unique_customer_id` as the customer counts. As shown above, there are a total of 99441 orders and 96096 unique customers in this dataset, meaning that 3345 orders were made by returning customers.

In [ ]:
# First 5 rows of the orders dataset
sql("""
"SELECT * FROM orders
"LIMIT 5
""", conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
# First 5 rows of the reviews dataset
sql("""
"SELECT * FROM reviews 
"LIMIT 5
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
# Total orders in the `orders` table
sql("""
SELECT COUNT(order_id) 
FROM orders
""", conn)

,COUNT(order_id)
0,99441


In [ ]:
# Total reviews and orders in the `reviews` table
sql("""
SELECT COUNT(review_id) AS total_reviews, 
COUNT(DISTINCT review_id) AS review_id_unique, 
COUNT(order_id) AS total_orders, 
COUNT(DISTINCT order_id) AS order_id_unique 
FROM reviews
""", conn)

,total_reviews,review_id_unique,total_orders,order_id_unique
0,99224,98410,99224,98673


To answer my primary question, we will need a dataset that has both the review scores (which lives in the `reviews` table) and the delivery dates (which lives in the `orders` dataset). The key variable for merging these 2 tables is `order_id`, and it is important to check the row counts and uniqueness of this variable in the 2 tables, as it would not only help me decide the table join method, but could also prevent future fanout problems.

While the count of `order_id` in the two tables doesn't match, this is totally reasonable since the customers don't always leave a review. And this problem could be address simply by giving the no-review orders an NA with LEFT JOIN.

However, we can see that in the `reviews` table, the number of total ids and unique ids doesn't match, suggesting that there are multiple review_id and order_id has more than one row of entry. This is where the fan-out problem could occur without extra care. So before deciding on how to handle this, I'd like to take a closer look on the reviews that share the same ids.

## Reviews sharing the same order_id

In [8]:
sql("""
SELECT order_id, COUNT(*) AS review_count
FROM reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY review_count DESC
LIMIT 10;
""", conn)

,order_id,review_count
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2
5,ff850ba359507b996e8b2fbb26df8d03,2
6,ff763b73e473d03c321bcd5a053316e8,2
7,fe041ba1c9f54016432fa6ee91709dbc,2
8,fd95ae805c63c534f1a64589e102225e,2
9,fd61441ba2a7b57e6342862e779b10b0,2


In [ ]:
# Inspect reviews with row[0] order_id
sql("""
SELECT *
FROM reviews
WHERE order_id = 'df56136b8031ecd28e200bb18e6ddb2e'
ORDER BY review_creation_date;
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,72a1098d5b410ae50fbc0509d26daeb9,df56136b8031ecd28e200bb18e6ddb2e,5,None,None,2017-02-07 00:00:00,2017-02-10 10:46:09
1,c444278834184f72b1484dfe47de7f97,df56136b8031ecd28e200bb18e6ddb2e,5,None,None,2017-02-08 00:00:00,2017-02-14 13:58:48
2,44f3e54834d23c5570c1d010824d4d59,df56136b8031ecd28e200bb18e6ddb2e,5,None,None,2017-02-09 00:00:00,2017-02-09 09:07:28


In [ ]:
# Inspect reviews with row[2] order_id
sql("""
SELECT *
FROM reviews
WHERE order_id = '8e17072ec97ce29f0e1f111e598b0c85'
ORDER BY review_creation_date;
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,67c2557eb0bd72e3ece1e03477c9dff5,8e17072ec97ce29f0e1f111e598b0c85,1,None,Entregou o produto errado.,2018-04-07 00:00:00,2018-04-08 22:48:27
1,2d6ac45f859465b5c185274a1c929637,8e17072ec97ce29f0e1f111e598b0c85,1,None,Comprei 3 unidades do produto vieram 2 unidades que não corresponde com o que comprei. Devido a minha opinião é negativa com relação a esse vendedor pois não não cumpriu com o prometido na venda.,2018-04-07 00:00:00,2018-04-07 21:13:05
2,6e4c4086d9611ae4cc0cc65a262751fe,8e17072ec97ce29f0e1f111e598b0c85,1,None,"Embora tenha entregue dentro do prazo, não enviou o produto que comprei.",2018-04-14 00:00:00,2018-04-16 11:37:31


In [ ]:
# Inspect reviews with row[3] order_id
sql("""
SELECT *
FROM reviews
WHERE order_id = '03c939fd7fd3b38f8485a0f95798f1f6'
ORDER BY review_creation_date;
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,405eb2ea45e1dbe2662541ae5b47e2aa,03c939fd7fd3b38f8485a0f95798f1f6,3,None,Seria ótimo se tivesem entregue os 3 (três) pedidos de uma única vez.,2018-03-06 00:00:00,2018-03-06 19:50:32
1,b04ed893318da5b863e878cd3d0511df,03c939fd7fd3b38f8485a0f95798f1f6,3,None,"Um ponto negativo que achei foi a cobrança de 3 taxas de entregas, sendo que comprei os 3 produtos iguais numa só compra.\r\nE mesmo comprando os produtos juntos, chegaram separados.",2018-03-20 00:00:00,2018-03-21 02:28:23
2,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,03c939fd7fd3b38f8485a0f95798f1f6,4,None,NaN,2018-03-29 00:00:00,2018-03-30 00:29:09


In [ ]:
# Inspect reviews with row[7] order_id
sql("""
SELECT *
FROM reviews
WHERE order_id = 'fe041ba1c9f54016432fa6ee91709dbc'
ORDER BY review_creation_date;
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,117b4a19af2b247561f0075f73d6eea1,fe041ba1c9f54016432fa6ee91709dbc,5,None,NaN,2017-12-20 00:00:00,2017-12-20 22:04:37
1,9da0796858da97acd8b67578388a3241,fe041ba1c9f54016432fa6ee91709dbc,5,None,"O cartucho está estourado (vazando), já contatei a empresa. Foi-me informado que haverá a troca!",2017-12-20 00:00:00,2017-12-20 22:03:18


For reviews with the same `order_id`, some only differs in `review_id`, `review_creation_date`, and `review_answer_timestamp` (e.g., order_id [0]), while some also differs in `review_comment_message` and `review_score` (e.g., order_id [2], [3], [7]), which signals that this isn't a system generated duplicate.

By looking at the `review_comment_message` and time stamps, I'm assuming that the reasons for repeated order_ids could be, but not limited to the following cases: (1) items in the same order were shipped seperately, (2) customers were commenting for different items in the same order, (3) customers can make an update about their experience (edit or add new review messages) for the same order/item multiple times.

For example, I translated the review comments of the case *order_id[3]* with Google Translate: 

- Row 0: "It would have been great if they had delivered the 3 (three) orders all at once."
- Row 1: "One downside I found was being charged 3 delivery fees, even though I bought 3 identical products in a single purchase. And despite buying the products together, they arrived separately."
- Row 2: No comments, but score improved to 4

This case clearly showed an example of the items in the same order being shipped seperately, and the customer updated/added reviews accordingly.

Another example is the case *order_id[2]*:

- Row 0: "Delivered the wrong product."
- Row 1: "I bought 3 units of the product, but 2 of the units that arrived did not match what I had purchased. Consequently, I have a negative opinion of this seller, as they did not fulfill the terms of the sale."
- Row 2: "Although delivery was on time, they did not send the product I bought."

The 3 review meesages were all about complaing the wrong product sent, each with a different date and time stamp, suggesting that it could be the customer continuously documenting this unresolved issue.

## Reviews sharing the same review_id

In [ ]:
sql("""
SELECT review_id, COUNT(*) AS row_count
FROM reviews
GROUP BY review_id
HAVING COUNT(*) > 1
LIMIT 20;
""", conn)

,review_id,row_count
0,00130cbe1f9d422698c812ed8ded1919,2
1,0115633a9c298b6a98bcbe4eee75345f,2
2,0174caf0ee5964646040cd94e15ac95e,2
3,017808d29fd1f942d97e50184dfb4c13,2
4,0254bd905dc677a6078990aad3331a36,2
5,0288d42bef3dfe36930740c9588a570f,2
6,02aa7f5f75e964e3c7efa59a1f515281,2
7,034528cb00b6bc981847acafbdf2ae0b,2
8,03a6a25db577d0689440933055111897,2
9,0467560f511c516ddaa54a60edb0c291,2


In [ ]:
# Inspect reviews with row[17] review_id
sql("""
SELECT *
FROM reviews
WHERE review_id = '08528f70f579f0c830189efc523d2182';
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,1,Produto errado,Entrega do produto diferente do solicitado\r\nAguardo orientação para a troca,2018-08-03 00:00:00,2018-08-06 00:09:52
1,08528f70f579f0c830189efc523d2182,03310aa823a66056268a3bab36e827fb,1,Produto errado,Entrega do produto diferente do solicitado\r\nAguardo orientação para a troca,2018-08-03 00:00:00,2018-08-06 00:09:52
2,08528f70f579f0c830189efc523d2182,53c71d3953507c6239ff73917ed358c9,1,Produto errado,Entrega do produto diferente do solicitado\r\nAguardo orientação para a troca,2018-08-03 00:00:00,2018-08-06 00:09:52


In [13]:
# Inspect reviews with row[6] review_id
sql("""
SELECT *
FROM reviews
WHERE review_id = '02aa7f5f75e964e3c7efa59a1f515281';
""", conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,02aa7f5f75e964e3c7efa59a1f515281,d8e17d5f7dacf0970d316e7c03e741e8,3,None,None,2017-07-27 00:00:00,2017-07-29 15:09:50
1,02aa7f5f75e964e3c7efa59a1f515281,db92613c074f00e53066388d48ad7512,3,None,None,2017-07-27 00:00:00,2017-07-29 15:09:50


As for reviews sharing the same `review_id`, I noticed from the two examples above that all columns are identical, except for `order_id`. While I don't have a reasonable guess for the cause of this right now, this phenomenon doesn't affect the joining of the tables, since `review_id` is not the key varirable for joining. For that, I will continue on other work, and come back to this in the future if needed.

## Handling rows with same order_id in the reviews table

Thinking back to my primary question, I'd only need the review scores to answer it at this stage, while the `review_comment_message` may help with getting more insights, it would be more efficient to focus on the `review_scores` for now, and come back to the `review_comment_message` later if needed.

For the rows with repeated `order_id`, I'm using the mean instead of median for the `review_score` column. This way, I could conserve the most information without worrying too much about outliers (given that the review score ranges 1-5 is relatively narrow, and that each group contains about 2-3 rows on average).

# Join Datasets